In [1]:
import easyvvuq as uq
from chaospy import DiscreteUniform
from easyvvuq.actions import ExecutePython, Actions

/home/mmachado/HPC/energyuq/.venv/lib/python3.12/site-packages/chaospy/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


As parameters of this example, we will use 2 integer inputs (frequency and number of threads).
Given enough runs, the problem should happen regardless of the limits ("min" and "max") for these numbers, though it will occur much faster if at least one of them ranges between 2 numbers close to each other (e.g., [1, 3], [1, 10], [4, 9])

In [2]:
QOI = 'energy' # our quantity of interest
max_frequency_levels = 3
max_threads = 32

refinement_iterations = 2

In [3]:
def define_params():
    return {
        "freq":   {"type": "integer", "min": 1, "max": max_frequency_levels, "default": max_frequency_levels}, # frequency levels
        "threads":{"type": "integer", "min": 1, "max": max_threads,          "default": max_threads,}  # available threads
    }

def define_vary():
    return {
        "freq":    DiscreteUniform(1, max_frequency_levels),
        "threads": DiscreteUniform(1, max_threads)
    }

# instead of using the more complex code that sets frequency and runs benchmarks with the specified number of threads,
# a quick generic example is used, since the error does not depend on what is being excuted. Could be anything
def run_example(params):
    return {
        QOI: (params['freq'] * params['threads']) ** 2
    }


In [4]:
campaign = uq.Campaign(
    name='example',
    params=define_params(),
    actions=Actions(ExecutePython(run_example))
)

The sampler must be SCSampler, since it's the one with dimension-adaptive capabilities

In [5]:
sampler = uq.sampling.SCSampler(
    vary=define_vary(),
    polynomial_order=1,
    quadrature_rule="C",
    sparse=True,
    midpoint_level1=True,
    dimension_adaptive=True
)

In [6]:
# set the sampler, and draw the first sample
campaign.set_sampler(sampler)
campaign.execute().collate(progress_bar=True)

100%|██████████| 1/1 [00:00<00:00, 1302.17it/s]


In [7]:
# Create an analysis class and run the analysis.
campaign.get_collation_result()
analysis = uq.analysis.SCAnalysis(sampler=sampler, qoi_cols=[QOI])
campaign.apply_analysis(analysis)

The errors will eventually occur while calling SCAnalysis.adapt_dimension during the refinement process

In [8]:
def refine_sampling_plan(number_of_refinements):
    """
    Copied from the tutorials
    """
    for i in range(number_of_refinements):
        # compute the admissible indices and run the ensemble
        sampler.look_ahead(analysis.l_norm)
        campaign.execute().collate(progress_bar=True)
    
        # accept one of the multi indices of the new admissible set
        data_frame = campaign.get_collation_result()
        analysis.adapt_dimension(QOI, data_frame)
        # When one of the dimensions "runs out" of available numbers, for instance:
        #       Dimension "freq" with available numbers [1, 2, 3] is "expanded" too much,
        #       the resulting sample has repeated elements [1, 1, 2, 3, 3]

In [9]:
# In this example, the error can be seen on the second refinement
refine_sampling_plan(refinement_iterations)

100%|██████████| 4/4 [00:00<00:00, 3121.92it/s]
0it [00:00, ?it/s]
/home/mmachado/HPC/energyuq/.venv/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/mmachado/HPC/energyuq/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
# In this example, the error can be seen on the second refinement
campaign.apply_analysis(analysis)